# 04 — 블랜딩 최종 (3-path OOF 기반 가중치 탐색)

01~03에서 각 경로가 저장한 **unit-level** 예측을 로드해서 가중치를 찾음.

- **입력**: `oof_unit.csv / val_unit.csv / test_unit.csv` (각 경로마다)
- **가중치 기준**: **train OOF unit RMSE** (val/test 비공개라 다른 기준 불가)
- **제약**: `w_i ≥ 0, Σw_i = 1`
- **출력**: `4_output/final/blend/{weights.json, final_val.csv, final_test.csv}`

**로컬 + Colab 양쪽 지원.** Colab 사용 시 `GDRIVE_FINAL_ID` + `GDRIVE_RESULTS_ID` 필요.

## 1. 환경 설정 + 모듈 import

### Colab 사용 가이드
1. `3_modeling/final/` zip → `GDRIVE_FINAL_ID`
2. `4_output/final/` 전체(01/02/03 결과) zip → `GDRIVE_RESULTS_ID`
   - 해제 시 `zit_only/`, `reg_only/{모델}/`, `zit_plus_reg/{모델}/` 구조 유지

In [ ]:
import os, sys, json

# ── Colab 사용 시에만 채울 것 ──
GDRIVE_FINAL_ID   = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'   # ★ final.zip 공유 ID
GDRIVE_RESULTS_ID = ''   # ★ 4_output/final/ 전체 zip 공유 ID (01/02/03 결과)

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/3_modeling/final/modules/hpo.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID가 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    # 01/02/03 결과 (blend 입력)
    _results_dir = '/content/project/4_output/final'
    if not os.path.exists(os.path.join(_results_dir, 'zit_only', 'oof_unit.csv')):
        assert GDRIVE_RESULTS_ID, 'GDRIVE_RESULTS_ID가 비어있음 — 01/02/03 결과 zip 필요'
        os.makedirs(_results_dir, exist_ok=True)
        os.system(f'gdown {GDRIVE_RESULTS_ID} -O /content/results.zip')
        os.system(f'unzip -qo /content/results.zip -d {_results_dir}')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all
from utils.evaluate import rmse

MODEL_ROOT = os.path.join(PROJECT_ROOT, "3_modeling")
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
from final.modules import blending

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

## 2. 블랜딩 설정

- `BLEND_PATHS` : 블랜딩에 포함할 경로 dict. `{name: 경로}` — 경로 하위에 `oof_unit.csv / val_unit.csv / test_unit.csv` 가 있어야 함
- 사용 안 할 경로는 주석 처리
- `BLEND_METHOD` : `'slsqp'` (수학해, 빠름) | `'optuna'` (탐색)

In [ ]:
EXP_ID = 'blend-001'
USER   = 'jh'

# ── 블랜딩 대상 경로 (주석 풀어 추가/제거) ──
# 기본값: 이미 실행 완료 가능성이 높은 2개(경로 A + 경로 C LGBM) 활성.
# 실제 사용 시 본인이 실행 완료한 경로만 남기고 나머지는 주석 처리하세요.
_BASE = os.path.join(OUTPUT_DIR, 'final')
BLEND_PATHS = {
    'zit':              os.path.join(_BASE, 'zit_only'),
    'reg_lgbm':         os.path.join(_BASE, 'reg_only', 'lgbm'),
    # 'reg_xgb':          os.path.join(_BASE, 'reg_only', 'xgb'),
    # 'reg_catboost':     os.path.join(_BASE, 'reg_only', 'catboost'),
    # 'reg_et':           os.path.join(_BASE, 'reg_only', 'et'),
    # 'reg_enet':         os.path.join(_BASE, 'reg_only', 'enet'),
    'zitreg_lgbm':      os.path.join(_BASE, 'zit_plus_reg', 'lgbm'),
    # 'zitreg_xgb':       os.path.join(_BASE, 'zit_plus_reg', 'xgb'),
    # 'zitreg_catboost':  os.path.join(_BASE, 'zit_plus_reg', 'catboost'),
    # 'zitreg_et':        os.path.join(_BASE, 'zit_plus_reg', 'et'),
    # 'zitreg_enet':      os.path.join(_BASE, 'zit_plus_reg', 'enet'),
}

BLEND_METHOD = 'slsqp'   # 'slsqp' | 'optuna'
N_TRIALS_OPTUNA = 300    # BLEND_METHOD='optuna'일 때만 사용

# ── 출력 경로 ──
OUT_DIR = os.path.join(_BASE, 'blend')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 경로 존재 확인 (없으면 안내하고 해당 경로만 자동 드랍) ──
_missing = []
for name, _p in list(BLEND_PATHS.items()):
    ok = all(os.path.exists(os.path.join(_p, csv))
             for csv in ('oof_unit.csv', 'val_unit.csv', 'test_unit.csv'))
    if not ok:
        _missing.append((name, _p))
        BLEND_PATHS.pop(name)
if _missing:
    print('[경고] 아래 경로의 unit CSV가 없어 자동 제외:')
    for name, _p in _missing:
        print(f'  - {name:20s}  ← {_p}')
    print('  → 01/02/03 해당 노트북을 먼저 실행해야 합니다.')

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'BLEND_METHOD: {BLEND_METHOD}')
print(f'블랜딩 대상 경로 ({len(BLEND_PATHS)}개):')
for name, _p in BLEND_PATHS.items():
    print(f'  - {name:20s}  ← {_p}')
print(f'OUT_DIR: {OUT_DIR}')

## 3. 데이터 로드 + 각 경로 unit 예측 로드

In [ ]:
# ── ys 로드 (train unit RMSE 평가 기준) ──
_, ys = load_all()
ys_train_unit = ys['train'].copy()

# y_train extreme clip (01~03과 동일 조건)
y_raw = ys_train_unit[TARGET_COL]
second_max = y_raw[y_raw < y_raw.max()].max()
ys_train_unit[TARGET_COL] = y_raw.clip(upper=second_max)
print(f'ys_train_unit: {ys_train_unit.shape}')

# ── 각 경로의 OOF/val/test unit 예측 로드 ──
train_preds = {}
val_preds   = {}
test_preds  = {}
for name, path in BLEND_PATHS.items():
    train_preds[name] = pd.read_csv(os.path.join(path, 'oof_unit.csv'))
    val_preds[name]   = pd.read_csv(os.path.join(path, 'val_unit.csv'))
    test_preds[name]  = pd.read_csv(os.path.join(path, 'test_unit.csv'))
    print(f'  {name:20s}  oof={train_preds[name].shape}  '
          f'val={val_preds[name].shape}  test={test_preds[name].shape}')

# ── 각 경로 단독 train RMSE (비교용) ──
print(f'\n── 각 경로 단독 train OOF RMSE ──')
_y = ys_train_unit.set_index(KEY_COL)[TARGET_COL]
for name, df in train_preds.items():
    p = df.set_index(KEY_COL)['pred'].loc[_y.index]
    r = float(np.sqrt(np.mean((p.values - _y.values) ** 2)))
    print(f'  {name:20s}  RMSE = {r:.6f}')

## 4. 블랜딩 (가중치 탐색 + val/test 적용)

- train OOF 기준으로 가중치 탐색 (val/test 비공개)
- 동일 가중치를 val/test 예측에 적용

In [ ]:
assert len(BLEND_PATHS) >= 2, (
    f'블랜딩엔 최소 2개 경로 필요 (현재 {len(BLEND_PATHS)}개). '
    '1개면 그 경로의 unit CSV를 그대로 제출하면 됩니다. '
    '주석 풀어 경로 추가하거나, 01/02/03 노트북 먼저 실행하세요.'
)

result = blending.fit_and_apply(
    train_preds=train_preds,
    val_preds=val_preds,
    test_preds=test_preds,
    y_train_unit=ys_train_unit,
    method=BLEND_METHOD,
    n_trials=N_TRIALS_OPTUNA,
)

weights      = result['weights']
train_blend  = result['train_blend']
val_blend    = result['val_blend']
test_blend   = result['test_blend']
train_rmse   = result['train_rmse']

print(f'\n[Blending 완료]')
print(f'  method:      {result["method"]}')
print(f'  train RMSE:  {train_rmse:.6f}')
print(f'  weights:')
for n, w in weights.items():
    print(f'    {n:20s}  {w:.4f}')

## 5. 최종 결과 저장

In [ ]:
# weights.json
meta = {
    'exp_id':      EXP_ID,
    'user':        USER,
    'method':      BLEND_METHOD,
    'paths':       {k: v for k, v in BLEND_PATHS.items()},
    'weights':     weights,
    'train_rmse':  train_rmse,
}
with open(os.path.join(OUT_DIR, 'weights.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

# 예측 CSV (KEY_COL + pred)
train_blend.to_csv(os.path.join(OUT_DIR, 'final_train_oof.csv'), index=False)
val_blend.to_csv(  os.path.join(OUT_DIR, 'final_val.csv'),       index=False)
test_blend.to_csv( os.path.join(OUT_DIR, 'final_test.csv'),      index=False)

# 제출 형식 (ufs_serial, health) — test 예측만
submission = test_blend.rename(columns={'pred': TARGET_COL})
submission.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)

print(f'[저장 완료] {OUT_DIR}')
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

## 6. 요약

In [ ]:
print(f'★ 04 블랜딩 최종')
print(f'  EXP_ID       : {EXP_ID}')
print(f'  method       : {BLEND_METHOD}')
print(f'  경로 수       : {len(BLEND_PATHS)}')
print(f'  train RMSE   : {train_rmse:.6f}  (블랜딩 후 train OOF 기준)')
print(f'  weights:')
for n, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f'    {n:20s}  {w:.4f}')
print(f'\n제출 파일: {os.path.join(OUT_DIR, "submission.csv")}')